# 07 — LoRA Finetuning

**Companion to [Chapter 07](../07-post-training.md).**

Full finetuning updates every weight and needs optimizer state for all of them
(~16 bytes per parameter). **LoRA** freezes the base model and learns a small
low-rank update instead — typically 1-2% of the parameters.

You will implement LoRA, inject it into a real model, finetune with it, and merge
the adapter back in.

In [1]:
import math, time, random
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(123)
random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


## 1. The idea, in arithmetic

For a weight matrix `W` of shape `(d, k)`, learn `dW = B @ A` with rank `r` small.

In [2]:
print(f"{'d x k':>12} {'full FT':>12} {'LoRA r=8':>10} {'LoRA r=16':>11} {'r=8 share':>11}")
for d in [768, 1024, 4096]:
    full = d * d
    print(f"{d:>5} x {d:<4} {full:>12,} {2*d*8:>10,} {2*d*16:>11,} {2*d*8/full*100:>10.2f}%")

print("\nThe insight: the UPDATE needed to adapt a model to a task has far lower")
print("intrinsic rank than the model itself.")

       d x k      full FT   LoRA r=8   LoRA r=16   r=8 share
  768 x 768       589,824     12,288      24,576       2.08%
 1024 x 1024    1,048,576     16,384      32,768       1.56%
 4096 x 4096   16,777,216     65,536     131,072       0.39%

The insight: the UPDATE needed to adapt a model to a task has far lower
intrinsic rank than the model itself.


## 2. Implementing LoRA

Two details that matter:

- **`B` initializes to zeros** so `B @ A = 0` and the adapted model is *exactly*
  the original at step 0. Training starts from a known-good point.
- **`scaling = alpha / rank`** is what you actually tune; alpha is usually 2x rank.

In [3]:
class LoRALinear(nn.Module):
    def __init__(self, base_layer, rank=8, alpha=16):
        super().__init__()
        self.base = base_layer
        for p in self.base.parameters():
            p.requires_grad = False                     # freeze the original

        in_f, out_f = base_layer.in_features, base_layer.out_features
        self.rank = rank
        self.A = nn.Parameter(torch.randn(rank, in_f) * 0.01)
        self.B = nn.Parameter(torch.zeros(out_f, rank))  # zeros -> identity at init
        self.scaling = alpha / rank

    def forward(self, x):
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.scaling

    def merged_weight(self):
        return self.base.weight + (self.B @ self.A) * self.scaling


base = nn.Linear(64, 64)
lora = LoRALinear(base, rank=4, alpha=8)

x = torch.randn(2, 64)
print("output identical to base at init:", torch.allclose(lora(x), base(x)))
print("(because B starts at zero, so B@A = 0)")

trainable = sum(p.numel() for p in lora.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in lora.parameters() if not p.requires_grad)
print(f"\ntrainable: {trainable:,}   frozen: {frozen:,}   ({trainable/frozen*100:.1f}%)")

output identical to base at init: True
(because B starts at zero, so B@A = 0)

trainable: 512   frozen: 4,160   (12.3%)


## 3. A small model to finetune

Same architecture as earlier notebooks, sized for CPU.

In [4]:
class MHA(nn.Module):
    def __init__(self, d, n_heads, ctx, dropout=0.0):
        super().__init__()
        self.n_heads, self.head_dim = n_heads, d // n_heads
        self.W_query = nn.Linear(d, d, bias=False)
        self.W_key   = nn.Linear(d, d, bias=False)
        self.W_value = nn.Linear(d, d, bias=False)
        self.out_proj = nn.Linear(d, d)
        self.register_buffer("mask", torch.triu(torch.ones(ctx, ctx), diagonal=1))

    def forward(self, x):
        b, n, d = x.shape
        q = self.W_query(x).view(b, n, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.W_key(x).view(b, n, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.W_value(x).view(b, n, self.n_heads, self.head_dim).transpose(1, 2)
        s = q @ k.transpose(2, 3) / k.shape[-1] ** 0.5
        s = s.masked_fill(self.mask.bool()[:n, :n], -torch.inf)
        o = (torch.softmax(s, -1) @ v).transpose(1, 2).contiguous().view(b, n, d)
        return self.out_proj(o)


class Block(nn.Module):
    def __init__(self, d, n_heads, ctx):
        super().__init__()
        self.att = MHA(d, n_heads, ctx)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)

    def forward(self, x):
        x = x + self.att(self.n1(x))
        return x + self.ff(self.n2(x))


class TinyGPT(nn.Module):
    def __init__(self, vocab, d=128, n_heads=4, n_layers=4, ctx=64):
        super().__init__()
        self.ctx = ctx
        self.tok = nn.Embedding(vocab, d)
        self.pos = nn.Embedding(ctx, d)
        self.blocks = nn.Sequential(*[Block(d, n_heads, ctx) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, vocab, bias=False)

    def forward(self, idx):
        n = idx.shape[1]
        x = self.tok(idx) + self.pos(torch.arange(n, device=idx.device))
        return self.head(self.norm(self.blocks(x)))


print("model defined")

model defined


## 4. Two corpora: pretraining and a new target style

The base model learns corpus A. Then LoRA adapts it to corpus B, which uses a
**different sentence structure** — a genuine distribution shift.

In [5]:
SUBJ = ["the cat", "the dog", "a bird", "the child"]
VERB = ["sits on", "jumps over", "looks at", "runs past"]
OBJ  = ["the mat", "the fence", "a chair", "the table"]

# corpus A: simple declarative sentences
corpus_a = " ".join(f"{random.choice(SUBJ)} {random.choice(VERB)} {random.choice(OBJ)}."
                    for _ in range(900))

# corpus B: a different structure - questions with a fixed answer form
corpus_b = " ".join(f"where is {random.choice(SUBJ)}? it is near {random.choice(OBJ)}!"
                    for _ in range(900))

chars = sorted(set(corpus_a + corpus_b))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data_a = torch.tensor(encode(corpus_a), dtype=torch.long)
data_b = torch.tensor(encode(corpus_b), dtype=torch.long)

print(f"vocab: {len(chars)} characters")
print(f"corpus A ({len(data_a):,} tokens): {corpus_a[:80]}")
print(f"corpus B ({len(data_b):,} tokens): {corpus_b[:80]}")

vocab: 26 characters
corpus A (25,041 tokens): the child runs past the mat. a bird runs past the table. a bird runs past a chai
corpus B (35,317 tokens): where is the cat? it is near the table! where is a bird? it is near the mat! whe


## 5. Pretrain on corpus A

In [6]:
CTX, BATCH = 64, 32

def get_batch(data):
    ix = torch.randint(len(data) - CTX - 1, (BATCH,))
    x = torch.stack([data[i:i + CTX] for i in ix])
    y = torch.stack([data[i + 1:i + CTX + 1] for i in ix])
    return x.to(device), y.to(device)


@torch.no_grad()
def eval_loss(model, data, iters=15):
    model.eval()
    tot = 0.0
    for _ in range(iters):
        x, y = get_batch(data)
        tot += F.cross_entropy(model(x).flatten(0, 1), y.flatten()).item()
    model.train()
    return tot / iters


def train(model, data, steps, lr):
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr)
    model.train()
    for step in range(steps):
        x, y = get_batch(data)
        opt.zero_grad()
        loss = F.cross_entropy(model(x).flatten(0, 1), y.flatten())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        opt.step()
    return model


torch.manual_seed(123)
model = TinyGPT(len(chars)).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"model parameters: {total_params:,}")
print(f"initial loss on A: {eval_loss(model, data_a):.4f}  (random init)")

t0 = time.time()
train(model, data_a, steps=400, lr=3e-3)
print(f"trained in {time.time() - t0:.0f}s")
print(f"\nloss on A (trained on it): {eval_loss(model, data_a):.4f}")
print(f"loss on B (never seen)   : {eval_loss(model, data_b):.4f}   <-- much worse")

model parameters: 806,656


initial loss on A: 3.3059  (random init)


trained in 24s



loss on A (trained on it): 0.1809


loss on B (never seen)   : 5.4296   <-- much worse


## 6. Inject LoRA and adapt to corpus B

Freeze everything, then wrap the query and value projections.

In [7]:
def apply_lora(module, rank=8, alpha=16, targets=("W_query", "W_value")):
    for name, child in module.named_children():
        if isinstance(child, nn.Linear) and name in targets:
            setattr(module, name, LoRALinear(child, rank, alpha))
        else:
            apply_lora(child, rank, alpha, targets)
    return module


import copy
lora_model = copy.deepcopy(model)
for p in lora_model.parameters():
    p.requires_grad = False                       # freeze EVERYTHING
apply_lora(lora_model, rank=8, alpha=16)
lora_model.to(device)

trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
print(f"total parameters    : {total_params:,}")
print(f"trainable with LoRA : {trainable:,}")
print(f"share trained       : {trainable / total_params * 100:.2f}%")
print()
print(f"loss on B before adaptation: {eval_loss(lora_model, data_b):.4f}")

total parameters    : 806,656
trainable with LoRA : 16,384
share trained       : 2.03%



loss on B before adaptation: 5.4240


In [8]:
t0 = time.time()
train(lora_model, data_b, steps=400, lr=3e-3)
print(f"LoRA finetuned in {time.time() - t0:.0f}s")

print(f"\nloss on B after LoRA : {eval_loss(lora_model, data_b):.4f}")
print(f"loss on A after LoRA : {eval_loss(lora_model, data_a):.4f}")
print(f"\n(base model, unchanged, still gets {eval_loss(model, data_a):.4f} on A)")

LoRA finetuned in 17s



loss on B after LoRA : 0.1176


loss on A after LoRA : 1.4030



(base model, unchanged, still gets 0.1822 on A)


### Compare with full finetuning

Same budget, all parameters trainable.

In [9]:
full_model = copy.deepcopy(model).to(device)
for p in full_model.parameters():
    p.requires_grad = True

t0 = time.time()
train(full_model, data_b, steps=400, lr=3e-4)
full_time = time.time() - t0

print(f"{'method':<18} {'trainable':>12} {'loss on B':>11} {'loss on A':>11}")
print("-" * 55)
print(f"{'base (no FT)':<18} {'-':>12} {eval_loss(model, data_b):>11.4f} {eval_loss(model, data_a):>11.4f}")
print(f"{'LoRA r=8':<18} {trainable:>12,} {eval_loss(lora_model, data_b):>11.4f} {eval_loss(lora_model, data_a):>11.4f}")
print(f"{'full finetune':<18} {total_params:>12,} {eval_loss(full_model, data_b):>11.4f} {eval_loss(full_model, data_a):>11.4f}")
print()
print("LoRA reaches nearly the same loss on the new task while training ~2% of")
print("the parameters. That is the headline result.")
print()
print("Note that BOTH methods forget task A badly (0.18 -> >1.2). LoRA does not")
print("automatically prevent catastrophic forgetting - a common myth. What it")
print("does give you is that the base weights are untouched, so you can DROP the")
print("adapter and instantly have the original model back. Verified below.")

method                trainable   loss on B   loss on A
-------------------------------------------------------


base (no FT)                  -      5.4397      0.1835


LoRA r=8                 16,384      0.1178      1.4118


full finetune           806,656      0.0978      1.2846

LoRA reaches nearly the same loss on the new task while training ~2% of
the parameters. That is the headline result.

Note that BOTH methods forget task A badly (0.18 -> >1.2). LoRA does not
automatically prevent catastrophic forgetting - a common myth. What it
does give you is that the base weights are untouched, so you can DROP the
adapter and instantly have the original model back. Verified below.


In [10]:
# The base weights inside the LoRA model were never modified.
# Disable the adapters and the original model is recovered exactly.
for m in lora_model.modules():
    if isinstance(m, LoRALinear):
        m.scaling = 0.0                      # switch the adapter off

print(f"LoRA model with adapters OFF, loss on A: {eval_loss(lora_model, data_a):.4f}")
print(f"original base model,        loss on A: {eval_loss(model, data_a):.4f}")
print("\nIdentical - the base is intact. The full finetune cannot do this:")
print(f"full finetune,              loss on A: {eval_loss(full_model, data_a):.4f}  (permanently changed)")

for m in lora_model.modules():               # restore
    if isinstance(m, LoRALinear):
        m.scaling = 16 / 8

LoRA model with adapters OFF, loss on A: 0.1817


original base model,        loss on A: 0.1826

Identical - the base is intact. The full finetune cannot do this:


full finetune,              loss on A: 1.3028  (permanently changed)


## 7. Generate with the adapted model

In [11]:
@torch.no_grad()
def generate(m, prompt, n=120, temperature=0.7):
    m.eval()
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    for _ in range(n):
        logits = m(idx[:, -CTX:])[:, -1, :]
        probs = torch.softmax(logits / temperature, dim=-1)
        idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
    return decode(idx[0].tolist())


print("BASE model (trained on declaratives):")
print("  ", generate(model, "the cat ", n=90))
print("\nLoRA model (adapted to questions):")
print("  ", generate(lora_model, "where is ", n=90))

BASE model (trained on declaratives):


   the cat looks at the table. the cat looks at a chair. the cat sits on the table. the cat sits on t

LoRA model (adapted to questions):


   where is the child? it is near the fence! where is a bird? it is near the table! where is the child


## 8. Merging the adapter

`W_new = W + (B @ A) * scaling` folds LoRA into the base weights, giving
**zero inference overhead**. Or keep adapters separate and hot-swap many
task-specific ones over one base model in memory.

In [12]:
lora_layer = lora_model.blocks[0].att.W_query
x = torch.randn(1, 8, 128, device=device)

with torch.no_grad():
    unmerged = lora_layer(x)

    merged = nn.Linear(128, 128, bias=False).to(device)
    merged.weight.copy_(lora_layer.merged_weight())
    merged_out = merged(x)

print("merged output matches unmerged:", torch.allclose(unmerged, merged_out, atol=1e-5))
print("max difference:", (unmerged - merged_out).abs().max().item())
print("\nAfter merging there is no extra matrix multiply at inference -")
print("the adapter is literally gone, absorbed into W.")

merged output matches unmerged: True
max difference: 1.3113021850585938e-06

After merging there is no extra matrix multiply at inference -
the adapter is literally gone, absorbed into W.


## 9. Practical guidance

| Setting | Guidance |
|---|---|
| **rank** | 8-16 for style/format, 32-64 for harder task shifts. Diminishing returns above. |
| **alpha** | Usually 2x rank. Only the ratio `alpha/rank` matters. |
| **targets** | Q and V projections is the classic choice; all linear layers works better but trains more. |
| **learning rate** | ~10x higher than full finetuning (LoRA params start near zero). |
| **QLoRA** | LoRA over a 4-bit quantized frozen base — finetune 70B on one 48 GB GPU. |

## Exercises

1. **Rank study.** Loop over rank in {1, 2, 4, 8, 16, 32}, retrain, and plot final loss vs trainable parameters. Find the knee.
2. Apply LoRA to **all** linear layers instead of just Q and V. How much better, and at what parameter cost?
3. Train two adapters on two different target corpora, then swap between them on one frozen base. That is multi-tenant serving in miniature.

**Next:** [08 — Modern Architecture Components](08_modern_components.ipynb)